In [3]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import csr_matrix
import os

In [4]:
active_subjects = pd.read_csv(os.path.join("..", "..","data", "active_subjects.csv"))
audiences = pd.read_csv(os.path.join("..", "..","data", 'audiencies.csv'))
institutions = pd.read_csv(os.path.join("..", "..","data", 'institutions.csv'))
passive_subjects = pd.read_csv(os.path.join("..", "..","data", 'passive_subjects.csv'))

In [5]:
df = active_subjects[["sujeto_pasivo_id", "Nombre completo"]].copy()

In [6]:
df

,sujeto_pasivo_id,Nombre completo
0,634851,Leslie Zapata
1,634851,Flora Flores
2,634851,Clara Blanco
3,634851,Conrado Blanco
4,634851,Yessica Sanches
...,...,...
1153805,391700,JAIME OSORIO
1153806,391700,LORENA OSORIO
1153807,391700,LORENA OSORIO
1153808,712954,Margarita Corrotea


In [7]:
df = active_subjects[["sujeto_pasivo_id", "Nombre completo"]].copy()

In [8]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [9]:
# --- Filtrar test ---
test_df = test_df[
    test_df["sujeto_pasivo_id"].isin(train_df["sujeto_pasivo_id"]) &
    test_df["Nombre completo"].isin(train_df["Nombre completo"])
].reset_index(drop=True)


In [10]:
user_enc = LabelEncoder()
item_enc = LabelEncoder()

train_users = user_enc.fit_transform(train_df["sujeto_pasivo_id"])
train_items = item_enc.fit_transform(train_df["Nombre completo"])

test_users = user_enc.transform(test_df["sujeto_pasivo_id"])
test_items = item_enc.transform(test_df["Nombre completo"])

In [11]:
n_users = len(user_enc.classes_)
n_items = len(item_enc.classes_)

train_data = np.ones(len(train_df), dtype=np.float32)
test_data = np.ones(len(test_df), dtype=np.float32)

train_mat = csr_matrix((train_data, (train_users, train_items)), shape=(n_users, n_items))
test_mat = csr_matrix((test_data, (test_users, test_items)), shape=(n_users, n_items))

In [12]:
item_mat = train_mat.T 

In [13]:
item_knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=20,
    n_jobs=-1
)
item_knn.fit(item_mat)

,n_neighbors,20
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,-1


In [14]:
def recall_at_k_item(model, train_mat, test_mat, k=10, users=None):
    hits, total = 0, 0
    item_mat = train_mat.T  # ítem-usuario
    n_items = item_mat.shape[0]
    users = users if users is not None else range(train_mat.shape[0])

    for u in users:
        if test_mat[u].nnz == 0:
            continue

        seen = train_mat[u].indices
        if len(seen) == 0:
            continue

        # Vecinos de los ítems vistos (en espacio de ítems)
        _, idx = model.kneighbors(item_mat[seen], n_neighbors=20)

        # Puntajes agregados por similitud
        rec_scores = np.asarray(item_mat[idx.flatten()].sum(axis=0)).ravel()
        rec_scores = np.zeros(n_items, dtype=np.float32)
        for item_ids in idx:
            rec_scores[item_ids] += 1.0

        # Excluir ítems ya vistos
        rec_scores[seen] = 0

        # Top-k recomendaciones
        rec_items = np.argsort(-rec_scores)[:k]

        # Métrica
        true_items = test_mat[u].indices
        hits += len(set(rec_items) & set(true_items))
        total += len(true_items)

    return hits / total if total > 0 else 0


In [15]:
subset = range(0, train_mat.shape[0], 5)  # ~5% de usuarios
recall = recall_at_k_item(item_knn, train_mat, test_mat, k=10, users=subset)
print(f"Item-KNN Recall@10 (subset): {recall:.4f}")

KeyboardInterrupt: 

In [16]:
def gini_coefficient(values):
    """Calcula la desigualdad en la distribución de recomendaciones."""
    array = np.sort(np.array(values).flatten())
    if np.amin(array) < 0: array -= np.amin(array)
    if len(array) == 0 or np.sum(array) == 0: return 0.0
    
    index = np.arange(1, array.shape[0] + 1)
    n = array.shape[0]
    return ((np.sum((2 * index - n  - 1) * array)) / (n * np.sum(array)))

In [17]:
from sklearn.metrics import ndcg_score
from collections import Counter
from tqdm import tqdm

def evaluar_itemknn_metricas(model, train_mat, test_mat, k=10, users_subset=None):
    print(f"=== EVALUANDO ITEM-KNN (K={k}) ===")
    
    # 1. Preparación
    # Necesitamos la matriz transpuesta (Item-User) para buscar vecinos de ítems
    item_user_mat = train_mat.T 
    n_items = item_user_mat.shape[0]
    
    # Si no defines subset, evalúa a todos (puede ser lento)
    users = users_subset if users_subset is not None else range(train_mat.shape[0])
    
    recalls = []
    ndcgs = []
    average_precisions = []
    
    # Para Gini: guardamos cuántas veces se recomienda cada ítem
    all_recommended_items = []

    # 2. Iteración por Usuario
    for u in tqdm(users, desc="Evaluando Usuarios"):
        # Validar si tiene datos de prueba
        if test_mat[u].nnz == 0:
            continue

        # Ítems vistos en entrenamiento (Historial)
        seen_indices = train_mat[u].indices
        if len(seen_indices) == 0:
            continue

        # --- A. GENERAR RECOMENDACIONES ---
        # Encontramos los K vecinos más cercanos para CADA ítem que el usuario vio
        # n_neighbors=20 (ajustable) para tener candidatos
        distances, indices = model.kneighbors(item_user_mat[seen_indices], n_neighbors=20)
        
        # Agregamos puntajes:
        # Convertimos distancia a similitud (Sim = 1 - Distancia, aprox para coseno)
        # sklearn devuelve distancia coseno, donde 0 es idéntico y 1 es opuesto.
        similarities = 1 - distances
        
        # Acumulador de puntajes para candidatos
        candidate_scores = {}
        
        for i in range(len(seen_indices)):
            for j in range(len(indices[i])):
                item_idx = indices[i][j]
                score = similarities[i][j]
                
                if item_idx not in candidate_scores:
                    candidate_scores[item_idx] = 0
                candidate_scores[item_idx] += score
        
        # Convertir a listas para ordenar
        candidates = np.array(list(candidate_scores.keys()))
        scores = np.array(list(candidate_scores.values()))
        
        # Filtrar: Quitar ítems que el usuario YA vio en train
        mask = ~np.isin(candidates, seen_indices)
        candidates = candidates[mask]
        scores = scores[mask]
        
        # Top-K
        if len(candidates) == 0:
            continue
            
        # Ordenar descendente por score
        top_k_idx = np.argsort(scores)[::-1][:k]
        top_k_items = candidates[top_k_idx]
        
        # Guardar para Gini
        all_recommended_items.extend(top_k_items)

        # --- B. CALCULAR MÉTRICAS ---
        true_items = test_mat[u].indices
        
        # 1. Recall@K
        hits = len(set(top_k_items) & set(true_items))
        recall = hits / len(true_items)
        recalls.append(recall)
        
        # Preparar vectores binarios para nDCG y MAP
        # relevance_binary: 1 si el ítem recomendado está en test, 0 si no
        relevance_binary = np.isin(top_k_items, true_items).astype(int)
        
        # 2. nDCG@K
        # Como solo tenemos relevancia binaria (1 o 0), pasamos [relevance_binary] como y_true y y_score
        # Pero sklearn pide y_true (ideal) y y_score (predicho).
        # Truco: y_true es "el ideal" (ordenado 1s primero), y_score es nuestro ranking (relevance_binary ya ordenado por score)
        if len(relevance_binary) > 0:
             # nDCG ideal vs real
             # Usamos la libreria o formula manual. Usaremos sklearn para robustez.
             # Sklearn ndcg_score espera (1, n_items). 
             # Construimos y_true ideal: todos los 1s posibles (limitado a K)
             ideal_relevance = np.zeros(k)
             ideal_relevance[:min(len(true_items), k)] = 1
             
             # Rellenar predicción hasta K
             pred_relevance = np.zeros(k)
             pred_relevance[:len(relevance_binary)] = relevance_binary
             
             ndcg = ndcg_score([ideal_relevance], [pred_relevance])
             ndcgs.append(ndcg)

        # 3. MAP@K (Average Precision)
        score_ap = 0.0
        num_hits = 0.0
        for i, p in enumerate(relevance_binary):
            if p == 1:
                num_hits += 1.0
                score_ap += num_hits / (i + 1.0)
        
        # MAP se divide por min(K, total_relevantes)
        divisor = min(k, len(true_items))
        if divisor > 0:
            average_precisions.append(score_ap / divisor)
        else:
            average_precisions.append(0.0)

    # --- C. RESULTADOS FINALES ---
    mean_recall = np.mean(recalls)
    mean_ndcg = np.mean(ndcgs)
    mean_map = np.mean(average_precisions)
    
    # Calcular Gini
    # Contamos frecuencia de cada ítem recomendado
    item_counts = Counter(all_recommended_items)
    # Lista de frecuencias (incluyendo ceros para items nunca recomendados si quisieras ser estricto,
    # pero para Gini de concentración de recomendaciones usamos solo los recomendados o rellenamos).
    # Para ser comparables con otros papers, usamos la distribución de lo recomendado.
    freqs = list(item_counts.values())
    # Rellenamos con 0 los items que nunca se recomendaron para ver la desigualdad real del catálogo
    zeros = [0] * (n_items - len(freqs))
    freqs.extend(zeros)
    
    gini = gini_coefficient(freqs)
    
    return mean_recall, mean_ndcg, mean_map, gini

# --- EJECUCIÓN ---

# Definir un subset para que no tarde horas (opcional)
subset_users = range(0, train_mat.shape[0], 5) # 20% de usuarios

recall, ndcg, map_k, gini = evaluar_itemknn_metricas(
    item_knn, train_mat, test_mat, k=10, users_subset=subset_users
)

print("\n📊 RESULTADOS ITEM-KNN:")
print("-" * 30)
print(f"✅ Recall@10: {recall:.4f}")
print(f"✅ nDCG@10:   {ndcg:.4f}")
print(f"✅ MAP@10:    {map_k:.4f}")
print(f"⚖️ Gini:      {gini:.4f} (Concentración)")
print("-" * 30)

=== EVALUANDO ITEM-KNN (K=10) ===


Evaluando Usuarios: 100%|██████████| 3971/3971 [06:50<00:00,  9.66it/s] 


📊 RESULTADOS ITEM-KNN:
------------------------------
✅ Recall@10: 0.0527
✅ nDCG@10:   0.7435
✅ MAP@10:    0.0376
⚖️ Gini:      0.9470 (Concentración)
------------------------------
